# STAF — Colab Launcher

Minimal launcher notebook for the [STAF Multimodal Deepfake Detector](https://github.com/RAJ-VARUN13/staf-forensics-multimodal).  
All training and evaluation logic lives in `train.py` and `evaluate.py` — this notebook only sets up the environment and calls those scripts.

In [ ]:
# Mount Google Drive (persistent checkpoint & dataset storage)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo & install dependencies
!git clone https://github.com/RAJ-VARUN13/staf-forensics-multimodal.git
%cd staf-forensics-multimodal
!pip install -q -r requirements.txt

In [ ]:
# Verify CUDA GPU
import torch
assert torch.cuda.is_available(), "No GPU — switch runtime: Runtime > Change runtime type > GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Extract dataset from Google Drive to Colab local SSD
# Update the zip path to match your Drive location
!unzip -qn /content/drive/MyDrive/FakeAVCeleb_v1.2.zip -d /content/

In [ ]:
# Preprocessing pipeline
!python -m staf.preprocessing.extract_frames  --config staf/configs/colab.yaml
!python -m staf.preprocessing.detect_faces    --config staf/configs/colab.yaml
!python -m staf.preprocessing.crop_faces      --config staf/configs/colab.yaml
!python -m staf.preprocessing.extract_audio   --config staf/configs/colab.yaml
!python -m staf.preprocessing.sync_modalities --config staf/configs/colab.yaml

In [ ]:
# Train
!python train.py --config staf/configs/colab.yaml --no_wandb

In [ ]:
# Evaluate latest checkpoint
import glob
ckpt = sorted(glob.glob("/content/results/*/checkpoints/best_model.pt"))[-1]
!python evaluate.py --checkpoint "{ckpt}" --split test

In [ ]:
# Save results to Google Drive
!cp -r /content/results/* /content/drive/MyDrive/STAF_experiments/